This code is for gif generating

In [ ]:
import os
# 使用 imageio.v2 以避免警告
import imageio.v2 as imageio
from PIL import Image

#########################################
# Part 1: 分别生成各组的 gif 动图
#########################################
def create_gif_from_directory(image_dir, output_gif, duration=500):
    """
    从指定目录中读取 png 图片（按文件名排序），合成为一个 gif 动图。

    参数：
        image_dir: str
            png 图片所在目录
        output_gif: str
            生成的 gif 文件名（可以包含路径）
        duration: int
            每帧之间的间隔（毫秒）
    """
    files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
    if not files:
        print(f"目录 {image_dir} 中没有找到 png 图片！")
        return

    images = []
    for fname in files:
        fpath = os.path.join(image_dir, fname)
        images.append(imageio.imread(fpath))

    imageio.mimsave(output_gif, images, duration=duration/1000)
    print(f"已保存 gif 动图：{output_gif}")

#########################################
# Part 2: 生成组合 gif 动图
#########################################
def create_composite_gif(dir_J, dir_F, dir_M, output_gif, duration=500):
    """
    生成组合 gif 动图，要求如下：
      - 只使用各组从 March 1 日开始的图片。
      - 设三个组的原始图片尺寸均相同，尺寸为 (W, H)。
      - 组合图画布尺寸为 (2*W, 2*H)。
      - 将 F 图放在右上角，M 图放在右下角；
      - 将 J 图放在左侧居中，即在画布左侧留出上下各半个 H 的空白，然后把 J 图粘贴到正中间。
    
    参数：
        dir_J, dir_F, dir_M: str
            分别为 J、F、M 组图片所在的目录
        output_gif: str
            输出 gif 文件路径
        duration: int
            每帧间隔（毫秒）
    """
    # 设置偏移量：假设 J 组图片从 1 月开始，需要跳过 1 月（31 天）和 2 月（28 天）的图片；
    # F 组图片从 2 月开始，需要跳过 2 月（28 天）的图片；M 组图片从 3 月开始
    offset_J = 31 + 28  # 59 帧
    offset_F = 28       # 28 帧
    offset_M = 0

    files_J = sorted([f for f in os.listdir(dir_J) if f.endswith('.png')])
    files_F = sorted([f for f in os.listdir(dir_F) if f.endswith('.png')])
    files_M = sorted([f for f in os.listdir(dir_M) if f.endswith('.png')])

    # 计算可用帧数：从各组偏移后剩余帧数的最小值
    n_frames = min(len(files_J) - offset_J, len(files_F) - offset_F, len(files_M) - offset_M)
    if n_frames <= 0:
        print("没有足够的图片生成组合 gif 动图，请检查各组图片数量及 offset 设置。")
        return

    composite_frames = []
    for i in range(n_frames):
        # 取出对应帧（加上偏移量）
        path_J = os.path.join(dir_J, files_J[i + offset_J])
        path_F = os.path.join(dir_F, files_F[i + offset_F])
        path_M = os.path.join(dir_M, files_M[i + offset_M])
        try:
            img_J = Image.open(path_J).convert("RGB")
            img_F = Image.open(path_F).convert("RGB")
            img_M = Image.open(path_M).convert("RGB")
        except Exception as e:
            print(f"读取图片出错：{e}")
            continue

        # 假设三个图尺寸相同
        width, height = img_J.size

        # 创建一个新画布，尺寸为 (2*width, 2*height)
        composite_width = 2 * width
        composite_height = 2 * height
        composite_img = Image.new("RGB", (composite_width, composite_height), color=(255, 255, 255))

        # 右上角放 F 图，位置 (width, 0)
        composite_img.paste(img_F, (width, 0))
        # 右下角放 M 图，位置 (width, height)
        composite_img.paste(img_M, (width, height))
        # 左侧居中放 J 图：水平位置为 0，垂直位置为 (2*height - height)//2 = height//2
        composite_img.paste(img_J, (0, height//2))

        composite_frames.append(composite_img)

    # 将组合帧保存为 gif 动图
    composite_frames[0].save(output_gif,
                              save_all=True,
                              append_images=composite_frames[1:],
                              duration=duration,
                              loop=0)
    print(f"已保存组合 gif 动图：{output_gif}")

#########################################
# 主程序入口
#########################################
if __name__ == "__main__":
    # Part 1: 生成各组单独的 gif 动图
    create_gif_from_directory('./plots/2d/J', 'J_animation.gif', duration=500)
    create_gif_from_directory('./plots/2d/F', 'F_animation.gif', duration=500)
    create_gif_from_directory('./plots/2d/M', 'M_animation.gif', duration=500)

    # Part 2: 生成组合 gif 动图（从 March 1 日开始的图片）
    dir_J = './plots/2d/J'
    dir_F = './plots/2d/F'
    dir_M = './plots/2d/M'
    create_composite_gif(dir_J, dir_F, dir_M, 'composite_animation.gif', duration=500)
